In [1]:
"""
================================================================================
  15_Detection_Baseline_Comparison.ipynb
  RT-DETR vs YOLOv8m vs WILLIE (Seg-to-Det Pipeline)
================================================================================

  PURPOSE: Compare dedicated detection models against WILLIE's novel
           segmentation-to-detection pipeline. Show that converting
           segmentation masks to bounding boxes outperforms trained detectors.

  PRODUCES (300 DPI, PNG + PDF):
    1. fig_det_map_bars            - mAP@0.5 comparison all models
    2. fig_det_grouped_metrics     - mAP50/mAP50-95/Precision/Recall grouped
    3. fig_det_rtdetr_vs_yolo      - Head-to-head RT-DETR vs YOLO
    4. fig_det_vs_willie        - Baselines vs WILLIE seg-to-det
    5. fig_det_paradigm_compare    - Traditional detection vs seg-to-det
    6. fig_det_efficiency          - AP vs parameters
    7. tbl_det_full_comparison     - Complete results table
================================================================================
"""

import os, warnings
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

PROJECT_ROOT = "."
FIGURES_DIR = os.path.join(PROJECT_ROOT, "artifacts/15_det_baseline_comparison/figures")
os.makedirs(FIGURES_DIR, exist_ok=True)

plt.rcParams.update({
    'font.size': 11, 'axes.titlesize': 14, 'axes.labelsize': 12,
    'figure.dpi': 100, 'savefig.dpi': 300, 'savefig.bbox': 'tight',
    'savefig.facecolor': 'white', 'axes.grid': True, 'grid.alpha': 0.3,
    'font.family': 'sans-serif',
})

def save_fig(fig, name, close=True):
    for ext in ['png', 'pdf']:
        fig.savefig(os.path.join(FIGURES_DIR, f"{name}.{ext}"),
                    dpi=300, bbox_inches='tight', facecolor='white')
    if close:
        plt.close(fig)
    print(f"  📈 {name} (.png + .pdf)")

print("=" * 80)
print("  Detection Baseline Comparison")
print("=" * 80)
print(f"  📁 Figures: {FIGURES_DIR}")

# ====================================================================
# RESULTS DATA
# ====================================================================

# Dedicated detection baselines (trained on WILLIE det split)
RTDETR = {
    'name': 'RT-DETR-L', 'type': 'baseline',
    'map50': 87.95, 'map50_95': 57.38,
    'precision': 88.15, 'recall': 89.38,
    'params': 32.0, 'train_time_min': 33.7,
    'epochs': 80, 'best_epoch': 65,
    'color': '#3498db',
}

YOLO = {
    'name': 'YOLOv8m', 'type': 'baseline',
    'map50': 91.22, 'map50_95': 59.56,
    'precision': 87.45, 'recall': 88.47,
    'params': 25.9, 'train_time_min': 30.0,
    'epochs': 80, 'best_epoch': 69,
    'color': '#2ecc71',
}

# WILLIE seg-to-det (no dedicated detection training)
WS_MINI = {
    'name': 'WS-MINI\n(Seg-to-Det)', 'type': 'willie',
    'map50': 86.70, 'map50_95': 0.0,
    'precision': 0.0, 'recall': 0.0,
    'params': 34.3, 'color': '#e67e22',
}

WS_BASE = {
    'name': 'WS-BASE\n(Seg-to-Det)', 'type': 'willie',
    'map50': 89.91, 'map50_95': 0.0,
    'precision': 0.0, 'recall': 0.0,
    'params': 520.4, 'color': '#e74c3c',
}

WS_XL = {
    'name': 'WS-XL\n(Seg-to-Det)', 'type': 'willie',
    'map50': 96.23, 'map50_95': 0.0,
    'precision': 96.23, 'recall': 94.64,
    'params': 762.5, 'color': '#c0392b',
}

ALL_MODELS = [RTDETR, YOLO, WS_MINI, WS_BASE, WS_XL]

print(f"\n  {'Model':<22s} {'AP@0.5':>8s} {'AP@50-95':>9s} {'Prec':>7s} {'Recall':>7s}")
print(f"  {'_'*55}")
for m in ALL_MODELS:
    n = m['name'].replace('\n', ' ')
    print(f"  {n:<22s} {m['map50']:>7.2f}% {m['map50_95']:>8.2f}% "
          f"{m['precision']:>6.2f}% {m['recall']:>6.2f}%")


# ====================================================================
# FIGURE 1: mAP@0.5 BARS (all models)
# ====================================================================
print("\n" + "=" * 70)
print("  FIGURE 1: mAP@0.5 Comparison")
print("=" * 70)

fig1, ax = plt.subplots(figsize=(12, 6))

names = [m['name'] for m in ALL_MODELS]
ap50s = [m['map50'] for m in ALL_MODELS]
colors = [m['color'] for m in ALL_MODELS]

bars = ax.bar(range(len(ALL_MODELS)), ap50s, color=colors,
              edgecolor='black', linewidth=0.5, alpha=0.85)
for bar, val in zip(bars, ap50s):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{val:.2f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.axvline(1.5, color='black', linestyle='--', lw=1.5, alpha=0.4)
ax.text(0.5, 78, 'Dedicated\nDetectors', ha='center', fontsize=10,
        style='italic', alpha=0.6)
ax.text(3, 78, 'willie\n(Seg-to-Det)', ha='center', fontsize=10,
        style='italic', alpha=0.6)

ax.set_xticks(range(len(ALL_MODELS)))
ax.set_xticklabels(names, fontweight='bold')
ax.set_ylabel('AP@0.5 (%)', fontweight='bold')
ax.set_title('Wound Detection: Dedicated Detectors vs Seg-to-Det Pipeline\n'
             'WILLIE-XL achieves 96.23% without any detection-specific training',
             fontweight='bold', fontsize=13, pad=15)
ax.set_ylim([75, 102])
save_fig(fig1, "fig_det_map_bars")


# ====================================================================
# FIGURE 2: GROUPED METRICS (RT-DETR vs YOLO only)
# ====================================================================
print("\n" + "=" * 70)
print("  FIGURE 2: Grouped Metrics (Baselines)")
print("=" * 70)

fig2, ax = plt.subplots(figsize=(10, 6))

metrics = ['mAP@0.5', 'mAP@50-95', 'Precision', 'Recall']
rt_vals = [RTDETR['map50'], RTDETR['map50_95'], RTDETR['precision'], RTDETR['recall']]
yo_vals = [YOLO['map50'], YOLO['map50_95'], YOLO['precision'], YOLO['recall']]

x = np.arange(len(metrics))
w = 0.3

b1 = ax.bar(x - w/2, rt_vals, w, label='RT-DETR-L', color=RTDETR['color'],
            edgecolor='white', alpha=0.85)
b2 = ax.bar(x + w/2, yo_vals, w, label='YOLOv8m', color=YOLO['color'],
            edgecolor='white', alpha=0.85)

for bars in [b1, b2]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.3,
                f'{h:.2f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')

# Mark winners
for i, (r, y) in enumerate(zip(rt_vals, yo_vals)):
    winner_x = i - w/2 if r > y else i + w/2
    winner_y = max(r, y) + 1.5
    ax.annotate('★', xy=(winner_x, winner_y), ha='center',
                fontsize=14, color='gold')

ax.set_xticks(x)
ax.set_xticklabels(metrics, fontweight='bold', fontsize=12)
ax.set_ylabel('Score (%)', fontweight='bold')
ax.set_title('RT-DETR-L vs YOLOv8m Head-to-Head\n'
             'YOLO wins mAP, RT-DETR wins Precision/Recall (2-2 tie)',
             fontweight='bold', fontsize=13, pad=15)
ax.legend(fontsize=11)
ax.set_ylim([50, 100])
save_fig(fig2, "fig_det_grouped_metrics")


# ====================================================================
# FIGURE 3: RT-DETR vs YOLO RADAR
# ====================================================================
print("\n" + "=" * 70)
print("  FIGURE 3: RT-DETR vs YOLO Radar")
print("=" * 70)

fig3, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))

categories = ['mAP@0.5', 'mAP@50-95', 'Precision', 'Recall']
N = len(categories)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

for model in [RTDETR, YOLO]:
    vals = [model['map50'], model['map50_95'], model['precision'], model['recall']]
    vals += vals[:1]
    ax.plot(angles, vals, 'o-', linewidth=2.5, markersize=7,
            color=model['color'], label=model['name'])
    ax.fill(angles, vals, alpha=0.1, color=model['color'])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontweight='bold', fontsize=11)
ax.set_ylim(50, 100)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=11)
ax.set_title('RT-DETR-L vs YOLOv8m\nDetection Profile',
             fontweight='bold', fontsize=14, pad=30)
save_fig(fig3, "fig_det_rtdetr_vs_yolo")


# ====================================================================
# FIGURE 4: BASELINES vs WILLIE
# ====================================================================
print("\n" + "=" * 70)
print("  FIGURE 4: Baselines vs WILLIE")
print("=" * 70)

fig4, ax = plt.subplots(figsize=(11, 6))

# Best baseline = YOLO at 91.22%
best_bl = YOLO['map50']

ws_names = ['Best Baseline\n(YOLOv8m)', 'WS-MINI', 'WS-BASE', 'WS-XL']
ws_vals = [best_bl, WS_MINI['map50'], WS_BASE['map50'], WS_XL['map50']]
ws_colors = ['#95a5a6', WS_MINI['color'], WS_BASE['color'], WS_XL['color']]

bars = ax.bar(range(4), ws_vals, color=ws_colors,
              edgecolor='black', linewidth=0.5, alpha=0.85, width=0.5)
for bar, val in zip(bars, ws_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{val:.2f}%', ha='center', fontweight='bold', fontsize=12)

# Delta annotations
for i in range(1, 4):
    delta = ws_vals[i] - best_bl
    color = '#27ae60' if delta > 0 else '#e74c3c'
    sign = '+' if delta > 0 else ''
    ax.text(i, ws_vals[i] - 2, f'{sign}{delta:.1f}%',
            ha='center', fontsize=10, color=color, fontweight='bold')

ax.axhline(best_bl, color='gray', linestyle=':', lw=1.5, alpha=0.5,
           label=f'Best baseline: {best_bl:.2f}%')
ax.set_xticks(range(4))
ax.set_xticklabels(ws_names, fontweight='bold')
ax.set_ylabel('AP@0.5 (%)', fontweight='bold')
ax.set_title('willie Seg-to-Det vs Best Detection Baseline\n'
             'XL exceeds YOLOv8m by +5.0% without detection-specific training',
             fontweight='bold', fontsize=13, pad=15)
ax.legend(fontsize=10)
ax.set_ylim([80, 102])
save_fig(fig4, "fig_det_vs_willie")


# ====================================================================
# FIGURE 5: PARADIGM COMPARISON
# ====================================================================
print("\n" + "=" * 70)
print("  FIGURE 5: Detection Paradigm Comparison")
print("=" * 70)

fig5, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))

# Left: Traditional detection pipeline
paradigms = ['Traditional\nDetection', 'willie\nSeg-to-Det']
best_trad = max(RTDETR['map50'], YOLO['map50'])
best_seg2det = WS_XL['map50']

bars = ax1.bar(paradigms, [best_trad, best_seg2det],
               color=['#3498db', '#e74c3c'],
               edgecolor='black', linewidth=0.5, alpha=0.85, width=0.5)
for bar, val in zip(bars, [best_trad, best_seg2det]):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f'{val:.2f}%', ha='center', fontweight='bold', fontsize=13)

delta = best_seg2det - best_trad
ax1.annotate(f'+{delta:.1f}%', xy=(1, best_seg2det + 2),
             ha='center', fontsize=14, color='#27ae60', fontweight='bold')
ax1.set_ylabel('Best AP@0.5 (%)', fontweight='bold')
ax1.set_title('Best Detection Approach\nper Paradigm', fontweight='bold')
ax1.set_ylim([85, 102])

# Right: What each paradigm requires
info = [
    ['', 'Traditional\nDetection', 'willie\nSeg-to-Det'],
    ['Training', 'Detection-specific\nloss + labels', 'Segmentation masks\n(reused)'],
    ['Architecture', 'Anchor/query-based\ndetection head', 'Connected components\non seg output'],
    ['AP@0.5', f'{best_trad:.2f}%', f'{best_seg2det:.2f}%'],
    ['Extra Params', 'Full detector\n(25-32M)', 'Zero\n(uses seg output)'],
]

ax2.axis('off')
table = ax2.table(cellText=info[1:], colLabels=info[0],
                  cellLoc='center', loc='center',
                  colWidths=[0.20, 0.35, 0.35])
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 2.0)

for j in range(3):
    table[0, j].set_facecolor('#2c3e50')
    table[0, j].set_text_props(color='white', fontweight='bold')
for i in range(1, 5):
    table[i, 2].set_facecolor('#d5f5e3')

ax2.set_title('Paradigm Comparison', fontweight='bold', fontsize=13, pad=15)

fig5.suptitle('Traditional Detection vs Seg-to-Det Pipeline',
              fontweight='bold', fontsize=14, y=1.03)
plt.tight_layout()
save_fig(fig5, "fig_det_paradigm_compare")


# ====================================================================
# FIGURE 6: EFFICIENCY
# ====================================================================
print("\n" + "=" * 70)
print("  FIGURE 6: Efficiency")
print("=" * 70)

fig6, ax = plt.subplots(figsize=(10, 7))

for m in ALL_MODELS:
    marker = 's' if m['type'] == 'willie' else 'o'
    size = 200 if m['type'] == 'willie' else 150
    ax.scatter(m['params'], m['map50'], s=size, c=m['color'],
               marker=marker, edgecolors='black', linewidths=1.5, zorder=3)
    n = m['name'].replace('\n', ' ')
    offset = 1.5 if m['map50'] < 95 else -2.0
    ax.annotate(f"{n}\n{m['map50']:.1f}%",
                xy=(m['params'], m['map50']),
                xytext=(m['params'], m['map50'] + offset),
                ha='center', fontsize=9, fontweight='bold')

ax.set_xlabel('Total Parameters (M)', fontweight='bold')
ax.set_ylabel('AP@0.5 (%)', fontweight='bold')
ax.set_title('Detection AP@0.5 vs Model Size\n'
             'circles = dedicated detectors, squares = WILLIE',
             fontweight='bold', fontsize=13, pad=15)
ax.set_xscale('log')
save_fig(fig6, "fig_det_efficiency")


# ====================================================================
# FIGURE 7: FULL TABLE
# ====================================================================
print("\n" + "=" * 70)
print("  FIGURE 7: Full Comparison Table")
print("=" * 70)

fig7, ax = plt.subplots(figsize=(16, 5.5))
ax.axis('off')

tbl_data = [
    ['Model', 'Type', 'Paradigm', 'Params', 'AP@0.5',
     'AP@50-95', 'Precision', 'Recall', 'Train Time']
]

for m in ALL_MODELS:
    n = m['name'].replace('\n', ' ')
    mtype = 'Multi-Task' if m['type'] == 'willie' else 'Single-Task'
    paradigm = 'Seg-to-Det' if m['type'] == 'willie' else 'Direct Det'
    ap95 = f"{m['map50_95']:.2f}%" if m['map50_95'] > 0 else '-'
    prec = f"{m['precision']:.2f}%" if m['precision'] > 0 else '-'
    rec = f"{m['recall']:.2f}%" if m['recall'] > 0 else '-'
    time_str = f"{m.get('train_time_min', 0):.0f} min" if m.get('train_time_min', 0) > 0 else 'N/A*'
    tbl_data.append([
        n, mtype, paradigm, f"{m['params']:.1f}M",
        f"{m['map50']:.2f}%", ap95, prec, rec, time_str
    ])

table = ax.table(cellText=tbl_data[1:], colLabels=tbl_data[0],
                 cellLoc='center', loc='center',
                 colWidths=[0.14, 0.10, 0.10, 0.08, 0.09, 0.09, 0.09, 0.09, 0.09])
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 1.8)

for j in range(9):
    table[0, j].set_facecolor('#2c3e50')
    table[0, j].set_text_props(color='white', fontweight='bold')

for i in range(1, len(tbl_data)):
    mtype = tbl_data[i][1]
    bg = '#d5f5e3' if mtype == 'Multi-Task' else 'white'
    for j in range(9):
        table[i, j].set_facecolor(bg)

# Highlight XL row
for j in range(9):
    table[5, j].set_text_props(fontweight='bold')

ax.set_title('Complete Detection Comparison\n'
             '*Seg-to-Det requires no detection-specific training '
             '(uses segmentation output)',
             fontweight='bold', fontsize=13, pad=25)
save_fig(fig7, "tbl_det_full_comparison")


# ====================================================================
# SUMMARY
# ====================================================================

fig_files = sorted([f for f in os.listdir(FIGURES_DIR)
                    if f.endswith(('.png', '.pdf'))])

print(f"\n{'='*80}")
print(f"  DETECTION BASELINE COMPARISON COMPLETE")
print(f"{'='*80}")
print(f"""
  KEY FINDINGS:
  - RT-DETR-L vs YOLOv8m: 2-2 TIE (YOLO wins mAP, RT-DETR wins P/R)
  - Best dedicated detector: YOLOv8m at 91.22% AP@0.5
  - WILLIE-XL Seg-to-Det: 96.23% AP@0.5 (+5.0% over best baseline)
  - Seg-to-Det requires ZERO detection-specific training
  - Novel finding: high-quality segmentation masks produce better
    bounding boxes than dedicated detection architectures

  {FIGURES_DIR}
  {len(fig_files)} files generated
""")
for f in fig_files:
    size = os.path.getsize(os.path.join(FIGURES_DIR, f))
    icon = '📈' if f.endswith('.png') else '📄'
    print(f"     {icon} {f}  ({size/1024:.1f} KB)")

  Detection Baseline Comparison
  📁 Figures: artifacts/15_det_baseline_comparison/figures

  Model                    AP@0.5  AP@50-95    Prec  Recall
  _______________________________________________________
  RT-DETR-L                87.95%    57.38%  88.15%  89.38%
  YOLOv8m                  91.22%    59.56%  87.45%  88.47%
  WS-MINI (Seg-to-Det)     86.70%     0.00%   0.00%   0.00%
  WS-BASE (Seg-to-Det)     89.91%     0.00%   0.00%   0.00%
  WS-XL (Seg-to-Det)       96.23%     0.00%  96.23%  94.64%

  FIGURE 1: mAP@0.5 Comparison
  📈 fig_det_map_bars (.png + .pdf)

  FIGURE 2: Grouped Metrics (Baselines)
  📈 fig_det_grouped_metrics (.png + .pdf)

  FIGURE 3: RT-DETR vs YOLO Radar
  📈 fig_det_rtdetr_vs_yolo (.png + .pdf)

  FIGURE 4: Baselines vs WILLIE
  📈 fig_det_vs_willie (.png + .pdf)

  FIGURE 5: Detection Paradigm Comparison
  📈 fig_det_paradigm_compare (.png + .pdf)

  FIGURE 6: Efficiency
  📈 fig_det_efficiency (.png + .pdf)

  FIGURE 7: Full Comparison Table
  📈 tbl_det_fu